In [2]:
import requests
import json
import os

In [3]:

from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
URL = "https://routes.googleapis.com/distanceMatrix/v2:computeRouteMatrix"


def get_route_matrix(location_test):

    origins = []
    destinations = []

    for lat, lon in location_test:

        point = {
            "waypoint": {
                "location": {
                    "latLng": {
                        "latitude": lat,
                        "longitude": lon
                    }
                }
            }
        }

        origins.append(point)
        destinations.append(point)

    headers = {
        "Content-Type": "application/json",

        "X-Goog-Api-Key": API_KEY,

        "X-Goog-FieldMask":
        "originIndex,destinationIndex,distanceMeters,duration,status"
    }

    body = {
        "origins": origins,

        "destinations": destinations,

        "travelMode": "DRIVE"
    }

    response = requests.post(
        URL,
        headers=headers,
        json=body
    )

    return response.json()

In [4]:
location_test = [

    (17.3850, 78.4867),

    (17.3920, 78.4910),

    (17.472684, 78.468485)

]

In [5]:
response = get_route_matrix(location_test)

response

[{'originIndex': 1, 'destinationIndex': 1, 'status': {}, 'duration': '0s'},
 {'originIndex': 0, 'destinationIndex': 0, 'status': {}, 'duration': '0s'},
 {'originIndex': 2, 'destinationIndex': 2, 'status': {}, 'duration': '0s'},
 {'originIndex': 1,
  'destinationIndex': 0,
  'status': {},
  'distanceMeters': 2567,
  'duration': '579s'},
 {'originIndex': 2,
  'destinationIndex': 0,
  'status': {},
  'distanceMeters': 12668,
  'duration': '2256s'},
 {'originIndex': 0,
  'destinationIndex': 1,
  'status': {},
  'distanceMeters': 1523,
  'duration': '325s'},
 {'originIndex': 0,
  'destinationIndex': 2,
  'status': {},
  'distanceMeters': 13165,
  'duration': '2120s'},
 {'originIndex': 1,
  'destinationIndex': 2,
  'status': {},
  'distanceMeters': 13087,
  'duration': '2113s'},
 {'originIndex': 2,
  'destinationIndex': 1,
  'status': {},
  'distanceMeters': 12537,
  'duration': '2276s'}]

In [6]:
for i in response:
    print(i)

{'originIndex': 1, 'destinationIndex': 1, 'status': {}, 'duration': '0s'}
{'originIndex': 0, 'destinationIndex': 0, 'status': {}, 'duration': '0s'}
{'originIndex': 2, 'destinationIndex': 2, 'status': {}, 'duration': '0s'}
{'originIndex': 1, 'destinationIndex': 0, 'status': {}, 'distanceMeters': 2567, 'duration': '579s'}
{'originIndex': 2, 'destinationIndex': 0, 'status': {}, 'distanceMeters': 12668, 'duration': '2256s'}
{'originIndex': 0, 'destinationIndex': 1, 'status': {}, 'distanceMeters': 1523, 'duration': '325s'}
{'originIndex': 0, 'destinationIndex': 2, 'status': {}, 'distanceMeters': 13165, 'duration': '2120s'}
{'originIndex': 1, 'destinationIndex': 2, 'status': {}, 'distanceMeters': 13087, 'duration': '2113s'}
{'originIndex': 2, 'destinationIndex': 1, 'status': {}, 'distanceMeters': 12537, 'duration': '2276s'}


In [ ]:
def extract_duration_matrix(
    response,
    num_locations
):

    matrix = [

        [0 for _ in range(num_locations)]

        for _ in range(num_locations)
    ]

    for item in response:

        origin = item["originIndex"]

        destination = item["destinationIndex"]

        duration_seconds = int(
            item["duration"].replace("s", "")
        )

        duration_minutes = round(
            duration_seconds / 60,
            2
        )

        matrix[origin][destination] = (
            duration_minutes
        )

    return matrix

In [8]:
duration_matrix = extract_duration_matrix(
    response,
    len(location_test)
)

duration_matrix

[[0.0, 5.42, 35.33], [9.65, 0.0, 35.22], [37.6, 37.93, 0.0]]

In [9]:
def nearest_neighbor_optimizer(
    duration_matrix
):

    num_locations = len(duration_matrix)

    visited = [False] * num_locations

    # Start from first location
    current = 0

    route = [current]

    visited[current] = True

    total_duration = 0

    for _ in range(num_locations - 1):

        nearest = None

        nearest_duration = float("inf")

        for next_location in range(num_locations):

            if not visited[next_location]:

                duration = duration_matrix[
                    current
                ][next_location]

                if duration < nearest_duration:

                    nearest_duration = duration

                    nearest = next_location

        route.append(nearest)

        visited[nearest] = True

        total_duration += nearest_duration

        current = nearest

    return {

        "optimized_route": route,

        "total_duration_min": round(
            total_duration,
            2
        )
    }

In [10]:
result = nearest_neighbor_optimizer(
    duration_matrix
)

result

{'optimized_route': [0, 1, 2], 'total_duration_min': 40.64}

In [11]:
location_names = [

    "Store_A",

    "Store_B",

    "Store_C"

]

optimized_route_names = [

    location_names[i]

    for i in result["optimized_route"]
]

optimized_route_names

['Store_A', 'Store_B', 'Store_C']

In [12]:
print(
    "Optimized Route:\n"
)

print(
    " → ".join(optimized_route_names)
)

print(
    f"\nTotal Travel Time: "
    f"{result['total_duration_min']} mins"
)

Optimized Route:

Store_A → Store_B → Store_C

Total Travel Time: 40.64 mins
